
###### 05_model_training

###### Purpose

Purpose of this notebook is to use the Gold Delta table to train multiple machine learning models, log experiments to MLflow, and save model performance results for comparison.


###### Input

- Gold Delta Table

######  Output


- MLflow experiment containing:
    - Parameters

    - Metrics

    - Logged model

    - Signature

    - Input Example

- Delta table:
    - dbw_agentic_ai_dev.telco_ai.model_training_results


- Models Trained:
    - Logistic Regression

    - Decision Tree

    - Random Forest

######  Architecture

```text
    
Read Gold Delta Table
        ↓
Define Features and Target
        ↓
Build Preprocessing Pipeline
        ↓
Split Training and Test Data
        ↓
Train Candidate Models
        ↓
Evaluate Model Performance
        ↓
Log Experiments to MLflow
        ↓
Save Model Comparison Results


```


###### Skills Covered

- Delta Lake

- Feature Preprocessing

- Scikit-learn Pipelines

- MLflow 

- Model Training

- Model Evaluation

###### Section 0 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 1 :  Import Libraries and Configure MLflow

In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.impute import SimpleImputer
from mlflow.models.signature import infer_signature

user_name = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

experiment_path = f"/Users/{user_name}/telco_churn_experiment"

mlflow.set_experiment(experiment_path)

#print(experiment_path)

###### Section 2 :  Load Gold data

In [0]:
gold_df = spark.table(GOLD_TABLE)
df = gold_df.toPandas()

###### Section 3 :  Prepare Training Data

In [0]:

X = df.drop(columns=["customerID", "Churn", "Churn_Flag"])
y = df["Churn_Flag"]

###### Section 4 :  Build preprocessing pipelines

In [0]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [col for col in X.columns if col not in num_cols]

print("Numeric Columns")
print(num_cols)

print("Categorical Columns")
print(cat_cols)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ]
)

###### Section 4 : Train/test split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

###### Section 5 : Define candidate models

In [0]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

###### Section 6 : Train and Log Models

In [0]:
results = []

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):

        print(f"Training {model_name}...")
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        if hasattr(pipeline.named_steps["model"], "predict_proba"):
            y_prob = pipeline.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_prob)
        else:
            roc_auc = None

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        run_id = mlflow.active_run().info.run_id

        mlflow.log_param("model_name", model_name)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        if roc_auc is not None:
            mlflow.log_metric("roc_auc", roc_auc)

        # Create sample input for signature
        input_example = X_train.head(5)

        # Get sample prediction output
        prediction_example = pipeline.predict(input_example)

        # Infer model signature
        signature = infer_signature(
            input_example,
            prediction_example
        )

        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            signature=signature,
            input_example=input_example
        )
       
        results.append({
            "model": model_name,
            "run_id": run_id,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": roc_auc
        })

results_df = pd.DataFrame(results)
display(results_df)



###### Section 6 : Save model comparison results

In [0]:
spark.createDataFrame(results_df).write.format("delta").mode("overwrite").saveAsTable(MODEL_RESULTS_TABLE)

###### Section 7 : Check model training results

In [0]:
%sql
SELECT * FROM  dbw_agentic_ai_dev.telco_ai.model_training_results

###### Notebook Summary

- Loaded the Gold Delta table and converted it to a Pandas DataFrame for Scikit-learn model training.

- Performed a train/test split and trained three classification models.

- Dynamically retrieved the Databricks user and created MLflow experiment.

- Logged model parameters, evaluation metrics, signatures, input examples, and model artifacts to MLflow.

- Appended results to the delta table - dbw_agentic_ai_dev.telco_ai.model_training_results.

###### Key Learnings

- Importance of MLflow Experiments

- Difference between MLflow Run and Experiment

- Purpose of artifact_path

- Purpose of infer_signature()

- Importance of input_example

- Difference between Parameters and Metrics

- Why model artifacts contain MLmodel, model.pkl and environment files

- Importance of using a preprocessing pipeline to ensure consistent transformations during training and inference.

###### Notebook Conclusion

- Successfully trained multiple customer churn classification models using a reusable preprocessing pipeline, logged every experiment to MLflow, and stored model comparison metrics for selecting the best model in the next notebook.

###### Next Notebook

06_model_evaluation

Purpose

- Compare all trained models, identify the best model and save the best model metadata for registration.